# Diabetes Risk Analysis and Prediction
### Python | Google Colab | Pandas | Seaborn | Scikit-learn

**Project objective:** Explore the diabetes-risk dataset, understand its features, visualize patterns, and (if the dataset contains a diabetes-risk label) train a machine-learning classification model.

**Important:** The screenshot shows a diabetes-risk dataset, not a traffic-congestion dataset. Use a project title about diabetes risk for this file. The screenshot only shows some columns; the code below first displays all column names so the correct target column can be selected.

## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files
from IPython.display import display

sns.set_theme()

## 2. Upload the CSV file

In [ ]:
uploaded = files.upload()
file_name = next(iter(uploaded))
print("Uploaded file:", file_name)

df = pd.read_csv(file_name)
print("Dataset loaded successfully!")
display(df.head())

## 3. Understand the dataset

In [ ]:
print("Shape (rows, columns):", df.shape)
print("\\nColumn names:")
print(df.columns.tolist())

print("\\nDataset information:")
df.info()

print("\\nNumerical summary:")
display(df.describe(include="all").T)

## 4. Check missing values and duplicate rows

In [ ]:
print("Missing values in each column:")
display(df.isnull().sum().to_frame("Missing Values"))

print("Duplicate rows:", df.duplicated().sum())

## 5. Clean the dataset

In [ ]:
# Remove exact duplicate rows
df = df.drop_duplicates().copy()

# Remove an ID/record column from analysis features, if present
id_cols = [c for c in df.columns if c.lower() in ["record_id", "recordid", "id"]]
print("ID columns detected:", id_cols)

# Keep a cleaned copy; missing numeric/categorical values will be handled
# separately in the analysis/model pipeline.
print("Cleaned dataset shape:", df.shape)

## 6. Explore numeric features

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
print("Numeric columns:", numeric_cols)

if numeric_cols:
    display(df[numeric_cols].hist(figsize=(14, 10), bins=20))
    plt.tight_layout()
    plt.show()
else:
    print("No numeric columns found.")

## 7. Explore categorical features

In [ ]:
categorical_cols = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
print("Categorical columns:", categorical_cols)

for col in categorical_cols:
    print("\\n", col)
    display(df[col].value_counts(dropna=False).head(15))

## 8. Correlation heatmap

In [ ]:
if len(numeric_cols) >= 2:
    plt.figure(figsize=(11, 7))
    sns.heatmap(df[numeric_cols].corr(), annot=True, cmap="coolwarm", fmt=".2f")
    plt.title("Correlation Between Numeric Features")
    plt.tight_layout()
    plt.show()
else:
    print("At least two numeric columns are needed for a correlation heatmap.")

## 9. Select the target column

A supervised prediction model needs a target/label column that tells us the known diabetes-risk category or outcome.

Run the next cell and inspect the printed column names. If your dataset has a label such as `Diabetes_Risk`, `DiabetesRisk`, `Risk_Level`, or a similar column, set `TARGET` to its exact spelling. **Do not use Age, BMI, or another input feature as the target.**

In [ ]:
print(df.columns.tolist())

# Change this to the exact target/label column name from your dataset.
TARGET = None  # Example: TARGET = "Diabetes_Risk"

# Optional: If the target is not obvious, inspect unique values in likely columns.

## 10. Train a machine-learning model (only if a valid target column exists)

In [ ]:
if TARGET is None:
    print("TARGET is not set. EDA is complete, but prediction cannot run yet.")
    print("Set TARGET above to the exact diabetes-risk label column, then run this cell again.")
elif TARGET not in df.columns:
    print(f"'{TARGET}' was not found. Check the exact spelling in df.columns.")
else:
    from sklearn.model_selection import train_test_split
    from sklearn.compose import ColumnTransformer
    from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
    from sklearn.impute import SimpleImputer
    from sklearn.pipeline import Pipeline
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

    data = df.dropna(subset=[TARGET]).copy()
    drop_cols = [TARGET] + id_cols
    X = data.drop(columns=drop_cols, errors="ignore")
    y = data[TARGET].astype(str)

    # Remove columns that contain no useful variation
    X = X.loc[:, X.nunique(dropna=False) > 1]

    if y.nunique() < 2:
        print("The target must contain at least two classes for classification.")
    elif X.shape[1] == 0:
        print("No usable feature columns remain after excluding target/ID.")
    else:
        num_features = X.select_dtypes(include=np.number).columns.tolist()
        cat_features = X.select_dtypes(exclude=np.number).columns.tolist()

        numeric_pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ])
        categorical_pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ])

        preprocessor = ColumnTransformer([
            ("numeric", numeric_pipeline, num_features),
            ("categorical", categorical_pipeline, cat_features)
        ])

        model = Pipeline([
            ("preprocessing", preprocessor),
            ("classifier", RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced"))
        ])

        stratify_y = y if y.value_counts().min() >= 2 else None
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=stratify_y
        )

        model.fit(X_train, y_train)
        predictions = model.predict(X_test)

        print("Accuracy:", round(accuracy_score(y_test, predictions), 4))
        print("\\nClassification report:")
        print(classification_report(y_test, predictions, zero_division=0))

        cm = confusion_matrix(y_test, predictions, labels=model.classes_)
        display = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=model.classes_)
        display.plot(xticks_rotation=45)
        plt.title("Confusion Matrix")
        plt.tight_layout()
        plt.show()

## 11. Project conclusion

Complete this section only after running the notebook and reviewing your actual outputs:

- The dataset contains **[write the row count]** rows and **[write the column count]** columns.
- The important features visible in the provided screenshot include Age, Gender, BMI, Blood Pressure, Fasting Glucose, Insulin, and Physical Activity (confirm the complete column names from `df.columns`).
- Missing values: **[write what `df.isnull().sum()` shows]**.
- The charts help describe the distribution of the dataset's features and relationships among numeric variables.
- If a valid target column exists, the Random Forest model provides a baseline classification result. Report the accuracy and classification report produced by the notebook.
- This is a synthetic dataset, so results should not be treated as medical diagnosis or clinical evidence.

## 12. Short presentation script

"Good morning Ma'am. My project title is Diabetes Risk Analysis and Prediction. The objective is to explore a diabetes-risk dataset, understand its features, and identify patterns in the data. I used Python in Google Colab with Pandas and NumPy for data handling, Matplotlib and Seaborn for visualization, and Scikit-learn for machine learning if a target label is available. First, I loaded the dataset, checked its shape, columns, missing values and duplicates. Then I explored the data using statistical summaries and graphs. Finally, if the dataset includes a diabetes-risk target column, I trained and evaluated a Random Forest classification model. This project is for academic data analysis only and is not a medical diagnosis tool. Thank you."